In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle

In [2]:
##Loading the dataset
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
##Preprocessing the data
##Dropping unnecessary columns
data.columns  # inspect column names

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [4]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, errors='ignore')
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [5]:
##geography and Gender are categorical variables, we need to encode them
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])
data ##gender is now encoded as 0 and 1, where 0 represents 'Female' and 1 represents 'Male'

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [6]:
##geography is also a categorical variable, we need to encode it using one-hot encoding as it has more than two categories 
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()  
geography_encoded = ohe.fit_transform(data[['Geography']])
geography_encoded

<10000x3 sparse matrix of type '<class 'numpy.float64'>'
	with 10000 stored elements in Compressed Sparse Row format>

In [7]:
ohe.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [8]:
geography_encoded_df = pd.DataFrame(geography_encoded.toarray(), columns=ohe.get_feature_names_out(['Geography']))
geography_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [9]:
##Combining the encoded geography columns with the original dataset and dropping the original Geography column
data = pd.concat([data, geography_encoded_df], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,France,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,France,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [10]:
##Droping the original Geography column
data = pd.concat([data.drop('Geography', axis=1), geography_encoded_df], axis=1)
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0,0.0,1.0,0.0


In [11]:
## Save the encoders and the scaler for future use in pickle files
with open('label_encoder.pkl', 'wb') as le_file:
    pickle.dump(le, le_file)
with open('onehot_encoder.pkl', 'wb') as ohe_file:
    pickle.dump(ohe, ohe_file)

In [12]:
## Divide the dataset into independent and dependent features
X = data.drop('Exited', axis=1)  # Independent features
y = data['Exited']  # Dependent feature

##Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##Scaling the features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
with open('scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)

In [14]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0,0.0,1.0,0.0


## ANN Implemantation

In [15]:
!pip uninstall tensorflow -y
!pip install tensorflow==2.10.1

  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)


You should consider upgrading via the 'C:\Users\shash\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [16]:
import tensorflow as tf
from tensorflow.keras.layers import Dense 
##Importing the Dense layer for building the neuron in hidden layers and output layer
from tensorflow.keras.models import Sequential 
##Importing the Sequential model for building the neural network
from tensorflow.keras.callbacks import EarlyStopping , TensorBoard 
##Importing EarlyStopping for preventing overfitting and TensorBoard for visualizing the training process
import datetime

In [17]:
X_train.shape ##Checking the shape of the training data to determine the input dimension for the neural network

(8000, 15)

In [18]:
##Building the neural network model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # Hidden layer 1 with 64 neurons and ReLU activation 
    Dense(32, activation='relu'),  # Hidden layer 2 with 32 neurons and ReLU activation
    Dense(1, activation='sigmoid')  # Output layer with 1 neuron and sigmoid activation for binary classification
]
)

In [19]:
model.summary() ##Summary of the model architecture

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                1024      
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 3,137
Trainable params: 3,137
Non-trainable params: 0
_________________________________________________________________


In [20]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow warnings

In [21]:
import tensorflow
opt  = tensorflow.keras.optimizers.Adam(learning_rate=0.01) ##Defining the optimizer with a learning rate of 0.001
loss = tensorflow.keras.losses.BinaryCrossentropy() ##Defining the loss function for binary classification

In [22]:
##Compiling the model with binary crossentropy loss function and Adam optimizer
model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

In [23]:
##Setup TensorBoard callback for visualizing the training process
from tensorflow.keras.callbacks import TensorBoard

log_dir = "logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


In [24]:
##Setting up EarlyStopping and TensorBoard callbacks
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [25]:
##Training the model with the training data and validating on the test data, using the defined callbacks
history = model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs=100,
    callbacks=[early_stopping_callback, tensorflow_callback]
)

Epoch 1/100
250/250 [==============================] - 1s 3ms/step - loss: 0.4077 - accuracy: 0.8292 - val_loss: 0.3451 - val_accuracy: 0.8510
Epoch 2/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3581 - accuracy: 0.8535 - val_loss: 0.3646 - val_accuracy: 0.8600
Epoch 3/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3476 - accuracy: 0.8579 - val_loss: 0.3547 - val_accuracy: 0.8625
Epoch 4/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3439 - accuracy: 0.8570 - val_loss: 0.3468 - val_accuracy: 0.8570
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3435 - accuracy: 0.8580 - val_loss: 0.3577 - val_accuracy: 0.8570
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3393 - accuracy: 0.8621 - val_loss: 0.3474 - val_accuracy: 0.8510


In [26]:
model.save('churn_model.h5') ##Saving the trained model in HDF5 format for future use

In [27]:
import tensorflow as tf
%load_ext tensorboard
%tensorboard --logdir logs/fit20260317-200440 --port 6008

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "C:\Users\shash\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\shash\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\shash\AppData\Local\Programs\Python\Python310\Scripts\tensorboard.exe\__main__.py", line 4, in <module>
  File "C:\Users\shash\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorboard\main.py", line 27, in <module>
    from tensorboard import default
  File "C:\Users\shash\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorboard\default.py", line 38, in <module>
    from tensorboard.plugins.graph import graphs_plugin
  File "C:\Users\shash\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorboard\plugins\graph\graphs_plugin.py", line 30, in <mod